In [ ]:
"""
IMPORTANT:
This project is designed to run exclusively on Google Colab.

It relies on Google Drive being mounted at:
    /content/drive/MyDrive/

Local execution is not supported.
"""


**INSTALLING LIBRARIES**

In [ ]:
!pip install langchain-community

**MOUNTS DRIVE**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # Mounts Google Drive into the Colab environment to access project files

# Changes the current working directory to the NeuroScape project folder in Google Drive
%cd /content/drive/MyDrive/NeuroScape

**INICIALIZING LLM**

In [ ]:
!pip install pdfminer.six
!pip install colab-xterm
%load_ext colabxterm
%xterm

In [ ]:
!ollama pull orca-mini:13b

**IMPORTING LIBRARIES**

In [ ]:
import os
import time
import openai
import tomllib
import numpy as np
import pandas as pd
import sys
from dotenv import load_dotenv, find_dotenv
from glob import glob

**INSERTING CITATIONS AND CALCULATING CITATION RATE**

In [ ]:
import pandas as pd
import requests
import time
from tqdm import tqdm

# Starting folders path
csv_directory = "/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience"
article_csv_file = "articles_merged_cleaned_filtered_clustered.csv"
input_path = f"{csv_directory}/{article_csv_file}"

# Receives csv file
df = pd.read_csv(input_path)

# Verifies availabe columns
print("CSV Columns:", df.columns)

# Get all citations through crossref
def get_citations_count(doi):
    if pd.isna(doi):
        return None
    url = f"https://api.crossref.org/works/{doi}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        # Some queries may not have 'is-referenced-by-count'
        return data['message'].get('is-referenced-by-count', 0)
    except Exception as e:
        print(f"Error DOI {doi}: {e}")
        return None

# Builds 'Citations' column with citation number, it is necessary to continue the pipeline
citations = []
for doi in tqdm(df['Doi'], desc="Fetching citations"):
    count = get_citations_count(doi)
    citations.append(count)
    time.sleep(0.1)  # Delay to avoid rate limit

df['Citations'] = citations

# Saves updated csv
output_path = f"{csv_directory}/articles_with_citations.csv"
df.to_csv(output_path, index=False)
print(f"Updated CSV saved in: {output_path}")


In [ ]:
import pandas as pd

# Starting folders path
csv_directory = "/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience"
article_csv_file = "articles_with_citations.csv"
input_path = f"{csv_directory}/{article_csv_file}"

# Receives csv file
df = pd.read_csv(input_path)

# Verifies availabe columns
print("CSV Columns:", df.columns)

# Last year to get citations
current_year = 2023

# Calculates article age
df['Article Age'] = current_year - df['Year'] + 1  # +1 para evitar divisão por zero no ano de publicação

# Validation to avoid invalid numbers or division by 0
df['Article Age'] = df['Article Age'].apply(lambda x: max(x, 1))

# Calculates Citation Rate
df['Citation Rate'] = df['Citations'] / df['Article Age']

# Verifies lines
print(df[['Title', 'Year', 'Citations', 'Article Age', 'Citation Rate']].head())

# Saves updated csv
output_path = f"{csv_directory}/articles_with_citation_rate.csv"
df.to_csv(output_path, index=False)
print(f"Updated CSV saved in: {output_path}")


**Cluster Definition**

In [ ]:
import os
import time
import tomllib
import numpy as np
import pandas as pd
import sys
from dotenv import load_dotenv, find_dotenv
from glob import glob

from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.parsing import parse_directories
from src.utils.hypersphere import get_centroids
from src.utils.load_and_save import load_embedding_shards, align_to_df
from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings

# Add the 'src' directory to the Python path
# This allows importing custom project modules (e.g., src.utils, src.classes) without import errors
sys.path.append('/content/drive/MyDrive/NeuroScape/src')

# Load environment keys
load_dotenv("/content/drive/MyDrive/NeuroScape/keys.env")

BASEPATH = os.environ['BASEPATH']

# Define prompt templates
DEFINITION_PROMPT = PromptTemplate(
    input_variables=["abstracts"],
    template="""
            You are provided with a list of scientific abstracts that belong to a specific research cluster. Your task is to:

1. **Identify the most frequent and relevant keywords and phrases** used in the abstracts within this cluster. The keywords should accurately characterize the research within the cluster and avoid wrong assertions.

2. **Provide a descriptive title** for the cluster that encapsulates the main themes and methodologies. The title should be concise and accurately reflect the content of the abstracts.

3. **Write a brief summary** (1-2 sentences) that describes the main themes and methodologies of the cluster. The summary should be clear and accurately reflect the content of the abstracts.

4. **Determine the main focus** of the cluster, whether it is on the themes or methodologies. A methodological focus is when a cluster focuses on method development or consistently applies a specific methodology. A thematic focus is when a cluster consistently studies a specific phenomenon.

**Output Format:**

Please present your findings in **JSON format** with the following structure:

  "Keywords": ["keyword1", "keyword2", "keyword3", ...],
  "Title": "Descriptive Cluster Title",
  "Description": "Brief summary of the cluster's main themes and methodologies.",
  "Focus": "The main focus of the cluster is either on the themes or methodologies."
```

**Instructions:**

* **Accuracy is crucial**: Ensure all information is directly supported by the provided abstracts. Do not include information not present in the abstracts or make external assumptions.

* **Clarity and Precision**: The keywords, title, and description should be clear and accurately reflect the content of the abstracts.

* **Conciseness**: Do not include any additional text or explanations beyond the specified JSON output. Do not generate more output than necessary.

Respond ONLY with valid JSON.
Do NOT include any prefix, explanation, or extra text.
The JSON must contain all fields: Keywords, Title, Description, Focus.

**Here are the abstracts:**
{abstracts}""",
)

if __name__ == '__main__':
    configurations = load_configurations()
    directories = parse_directories()

    required_fields = configurations['definition']['required_fields']

    # Substituting OpenAI -> Ollama
    llm = ChatOllama(
        model="orca-mini:13b",
        temperature=0.0
    )

    definition_chain = DEFINITION_PROMPT | llm | SimpleJsonOutputParser()

    # Load the CSV file
    csv_directory = os.path.join('/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience')
    article_csv_file = 'articles_with_citation_rate.csv'
    article_df = pd.read_csv(os.path.join(csv_directory, article_csv_file))

    # Define the output path
    cluster_csv_file = 'clusters_defined.csv'
    output_path = os.path.join(csv_directory, cluster_csv_file)

    if os.path.exists(output_path):
        cluster_definitions_df = pd.read_csv(output_path)
        cluster_definitions = cluster_definitions_df.to_dict('records')
        processed_clusters = set(cluster_definitions_df['Cluster ID'].values.tolist())
    else:
        cluster_definitions = []
        processed_clusters = set()

    # Load the embeddings and PMIDs
    shard_directory = os.path.join(
        BASEPATH, directories['internal']['intermediate']['hdf5']['neuro'])
    files = glob(os.path.join(shard_directory, '*.h5'))
    embeddings, pmids = load_embedding_shards(files)

    # Align the embeddings and PMIDs to the DataFrame
    aligned_embeddings, aligned_pmids = align_to_df(embeddings, pmids, article_df)

    # Free memory
    del embeddings
    del pmids

    # Get the centroids of the clusters
    centroids = get_centroids(aligned_embeddings, article_df['Cluster ID'].values)
    centroid_similarities = centroids.dot(centroids.T) - np.eye(centroids.shape[0])

    # Get cluster labels
    labels = article_df['Cluster ID'].values

    # Get the abstracts
    abstracts = article_df['Abstract'].values

    # Get the publication years of the articles
    article_years = article_df['Year'].values

    # Get the citation rates of the articles
    citation_rates = article_df['Citation Rate'].values

    # Get the type of the articles
    article_types = article_df['Type'].values

    print("Starting clusters processing...")

    for label, centroid in enumerate(centroids):

        if label in processed_clusters:
            continue

        print(f"Processing cluster {label}...")

        cluster_embeddings = aligned_embeddings[labels == label]
        cluster_abstracts = abstracts[labels == label]

        number_of_abstracts = min(
            1,
            len(cluster_abstracts))

        cluster_abstracts = get_abstract_strings(
            centroid,
            cluster_embeddings,
            cluster_abstracts,
            number_of_abstracts
        )

        chain_input = {"abstracts": cluster_abstracts}
        cluster_definition = safe_dictionary_extraction(
            required_fields,
            chain_input,
            definition_chain,
            2,
            2.0
        )

        if cluster_definition is None:
            print(f"Cluster {label} didn't generate a valid JSON, skipping...")
            continue

        # Join the keywords into a single string
        cluster_definition['Keywords'] = '; '.join(cluster_definition['Keywords'])

        cluster_definition['Cluster ID'] = label
        cluster_definition['Size'] = sum(labels == label)
        cluster_definition['Year First Article'] = article_years[labels == label].min()

        # Adding citation metric
        cluster_definition['MCR Research'] = np.median(
            citation_rates[(labels == label) & (article_types == 'Research')])
        cluster_definition['MCR Review'] = np.median(
            citation_rates[(labels == label) & (article_types == 'Review')])

        most_similar = np.argmax(centroid_similarities[label])
        cluster_definition['Most Similar Cluster'] = most_similar
        cluster_definition['Similarity'] = centroid_similarities[label, most_similar]

        cluster_definitions.append(cluster_definition)

    # Convert the cluster definitions to a DataFrame
    cluster_definitions_df = pd.DataFrame(cluster_definitions)

    # Reorder the columns
    cluster_definitions_df = cluster_definitions_df[[
        'Cluster ID', 'Title', 'Size', 'Year First Article', 'MCR Research',
        'MCR Review', 'Keywords', 'Description', 'Focus',
        'Most Similar Cluster', 'Similarity'
    ]]

    # Save the cluster definitions to a CSV file
    cluster_definitions_df.to_csv(output_path, index=False)


**Cluster Distinction**

In [ ]:
from src.utils.parsing import parse_directories
from src.utils.hypersphere import get_centroids
from src.utils.load_and_save import load_embedding_shards, align_to_df
from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings

from dotenv import load_dotenv, find_dotenv
from glob import glob
import os
import pandas as pd

from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

# Add the 'src' directory to the Python path
# This allows importing custom project modules (e.g., src.utils, src.classes) without import errors
sys.path.append('/content/drive/MyDrive/NeuroScape/src')

# Load environment keys
load_dotenv("/content/drive/MyDrive/NeuroScape/keys.env")
BASEPATH = os.environ['BASEPATH']

# Define prompt template
DEFINITION_PROMPT = PromptTemplate(
    input_variables=["cluster_a_abstracts", "cluster_b_abstracts"],
    template="""
You are provided with two sets of neuroscientific abstracts from Cluster A and Cluster B, which are similar in nature. Analyze the abstracts from both clusters and identify the most distinguishing features that separate Cluster A from Cluster B with high accuracy and conciseness.

**Instructions:**
- Return your findings **only** in JSON format.
- The JSON should contain a single field named `"Distinguishing Features"`.
- Do **not** include any other text, explanations, or comments.
- Do **not** include more than three features.
- Always contrast the features of Cluster A with Cluster B.
- Keep the feature descriptions short (1 or 2 sentences) and simple. Non-experts must be able to understand.

**Example Output:**
```json
{{
  "Distinguishing Features": "Feature 1 description; Feature 2 description; Feature 3 description"
}}

**Cluster A Abstracts:**
{cluster_a_abstracts}

**Cluster B Abstracts:**
{cluster_b_abstracts}
"""
)

if __name__ == '__main__':
    configurations = load_configurations()
    required_fields = configurations['distinction']['required_fields']
    directories = parse_directories()

    checkpoints_folder = os.path.join('/content/drive/MyDrive/NeuroScape/output/checkpoints/clusters/distinction')

    # Substituting OpenAI -> Ollama
    llm = ChatOllama(
        model="orca-mini:13b",
        temperature=0.0
    )

    distinction_chain = DEFINITION_PROMPT | llm | SimpleJsonOutputParser()

    # Load CSVs
    csv_directory = os.path.join('/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience')
    article_csv_file = 'articles_with_citation_rate.csv'
    cluster_csv_file = 'clusters_defined.csv'
    article_df = pd.read_csv(os.path.join(csv_directory, article_csv_file))
    print(f"Loaded articles: {len(article_df)}")

    input_file = os.path.join(csv_directory, cluster_csv_file)
    cluster_definitions_df = pd.read_csv(input_file)
    print(f"Loaded clusters: {len(cluster_definitions_df)}")

    output_file = input_file.replace('.csv', '_distinguished.csv')
    checkpoint_file = os.path.join(checkpoints_folder,
                                   'cluster_distinction_checkpoint.csv')

    if os.path.exists(checkpoint_file):
        cluster_distinctions_df = pd.read_csv(checkpoint_file)
        cluster_distinctions = cluster_distinctions_df.to_dict('records')
        processed_clusters = set(cluster_distinctions_df['Cluster ID'].values.tolist())
        print(f"Checkpoint found, processed clusters: {len(processed_clusters)}")
    else:
        cluster_distinctions = []
        processed_clusters = set()
        print("No checkpoint found, starting from scratch.")

    # Load embeddings
    shard_directory = os.path.join(
        BASEPATH, directories['internal']['intermediate']['hdf5']['neuro'])
    files = glob(os.path.join(shard_directory, '*.h5'))
    embeddings, pmids = load_embedding_shards(files)

    aligned_embeddings, aligned_pmids = align_to_df(embeddings, pmids, article_df)
    del embeddings, pmids

    centroids = get_centroids(aligned_embeddings, article_df['Cluster ID'].values)
    centroid_similarities = centroids.dot(centroids.T) - np.eye(centroids.shape[0])
    labels = article_df['Cluster ID'].values
    abstracts = article_df['Abstract'].values

    for label, centroid in enumerate(centroids):
        if label in processed_clusters:
            continue

        print(f"\Processing cluster {label}...")
        cluster_embeddings = aligned_embeddings[labels == label]
        cluster_abstracts = abstracts[labels == label]

        similar_cluster_label = cluster_definitions_df.loc[label, 'Most Similar Cluster']
        print(f"Most similar cluster: {similar_cluster_label}")
        similar_cluster_embeddings = aligned_embeddings[labels == similar_cluster_label]
        similar_cluster_abstracts = abstracts[labels == similar_cluster_label]
        similar_cluster_centroid = centroids[similar_cluster_label]

        number_of_abstracts_cluster = min(
            2 // 2,
            len(cluster_abstracts))
        number_of_abstracts_similar_cluster = min(
            2 // 2,
            len(similar_cluster_abstracts))

        print(f"   Using {number_of_abstracts_cluster} abstracts from cluster {label} and "
              f"{number_of_abstracts_similar_cluster} from cluster {similar_cluster_label}")

        cluster_abstracts = get_abstract_strings(similar_cluster_centroid,
                                                 cluster_embeddings,
                                                 cluster_abstracts,
                                                 number_of_abstracts_cluster)
        similar_cluster_abstracts = get_abstract_strings(
            centroid, similar_cluster_embeddings, similar_cluster_abstracts,
            number_of_abstracts_similar_cluster)

        chain_input = {
            "cluster_a_abstracts": cluster_abstracts,
            "cluster_b_abstracts": similar_cluster_abstracts
        }

        print("   Calling LLM for distinction...")
        cluster_distinction = safe_dictionary_extraction(
            required_fields, chain_input, distinction_chain,
            2, 2.0)

        print(f"   Result: {cluster_distinction}")

        cluster_distinction['Cluster ID'] = label
        cluster_distinction['Distinguishing Features'] = cluster_distinction[
            'Distinguishing Features'].replace(', ', '; ').replace(
                'A', f'{label}').replace('B', f'{similar_cluster_label}')
        cluster_distinctions.append(cluster_distinction)

        cluster_distinctions_df = pd.DataFrame(cluster_distinctions)
        cluster_distinctions_df.to_csv(checkpoint_file, index=False)
        print(f"Checkpoint saved: {checkpoint_file}")

    cluster_definitions_df['Distinguishing Features'] = cluster_distinctions_df['Distinguishing Features']
    cluster_definitions_df.to_csv(os.path.join(csv_directory, output_file), index=False)
    print(f"\nFinal file saved in: {output_file}")


**Open Questions (ordem correta do pipeline)**

In [ ]:
import os
import sys
from glob import glob
import pandas as pd
from tqdm import tqdm

from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.parsing import parse_directories
from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_review_text

from dotenv import load_dotenv

# Add the 'src' directory to the Python path
# This allows importing custom project modules (e.g., src.utils, src.classes) without import errors
sys.path.append('/content/drive/MyDrive/NeuroScape/src')

# Load environment keys
load_dotenv("/content/drive/MyDrive/NeuroScape/keys.env")
BASEPATH = os.environ['BASEPATH']
# Prompt template
QUESTIONS_PROMPT = PromptTemplate(
    input_variables=["cluster_title", "cluster_definition", "reviews"],
    template="""

    You are a domain expert in neuroscience. You have the combined text of several recent neuroscience review articles that all pertain to the same cluster entitled "{cluster_title}".
    {cluster_definition}
    Your task is to synthesize and prioritize the major open questions explicitly or implicitly mentioned in these articles.

    Below are the texts (or substantial excerpts) of several review articles that cover this cluster. Each article might mention multiple open questions, challenges, or research gaps. Please read them carefully and produce a consolidated list of the major open questions in this subdomain.

**Instructions:**
- Return your findings **only** in JSON format (valid JSON, no extra text).
- The JSON should contain a single field named `"Open Questions"`.
- Do **not** include any other text, explanations, or comments.
- Do **not** include more than five Open Questions.
- No lists: Provide the Open Questions as a single string separated by semicolons, not as a list.
- Highlight overlap: Prioritize questions that appear in multiple reviews.
- Avoid fabrications: Only list questions that are actually stated or strongly implied by the articles.
- Condense duplication: If multiple reviews mention the same open question, merge them into a single entry.
- Preserve specificity: Include enough detail to capture the essence of each open question.
- Be concise: Keep each open question short (2 or 3 sentences) and simple. Non-experts must be able to understand.

**Example Output:**
```json
{{
  "Open Questions": "Open question 1; Open question 2; Open question 3"
}}


**Reviews:**
{reviews}
""")

if __name__ == '__main__':
    configurations = load_configurations()
    required_fields = configurations['questions']['required_fields']
    directories = parse_directories()

    # LLM Ollama
    llm = ChatOllama(
        model="orca-mini:13b",
        temperature=0.0
    )
    questions_chain = QUESTIONS_PROMPT | llm | SimpleJsonOutputParser()

    # Directories and files
    checkpoint_path = os.path.join(BASEPATH, directories['internal']['checkpoints'])
    pdf_directory = os.path.join(BASEPATH, directories['internal']['reference']['pdfs'])
    csv_directory = os.path.join('/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience')
    cluster_csv_file = 'clusters_defined_distinguished.csv'

    cluster_definitions_df = pd.read_csv(os.path.join(csv_directory, cluster_csv_file))
    output_file = os.path.join(csv_directory, cluster_csv_file.replace('.csv', '_questions.csv'))
    checkpoint_file = os.path.join(checkpoint_path, 'questions_checkpoint.csv')

    if os.path.exists(checkpoint_file):
        cluster_questions_df = pd.read_csv(checkpoint_file)
        cluster_questions = cluster_questions_df.to_dict('records')
        processed_clusters = set(cluster_questions_df['Cluster ID'].values.tolist())
        print(f'{len(processed_clusters)} processed clusters.')
    else:
        cluster_questions = []
        processed_clusters = set()
        print("No checkpoint found, starting from scratch.")

    unique_clusters = sorted(cluster_definitions_df['Cluster ID'].values.tolist())

    for cluster in tqdm(unique_clusters):
        if cluster in processed_clusters:
            continue

        cluster_pdf_directory = os.path.join(pdf_directory, f'cluster_{str(cluster).zfill(3)}')
        pdf_files = glob(os.path.join(cluster_pdf_directory, '*.pdf'))

        cluster_title = cluster_definitions_df.loc[cluster, 'Title']
        cluster_definition = cluster_definitions_df.loc[cluster, 'Description']

        reviews = get_review_text(pdf_files)

        chain_input = {
            "cluster_title": cluster_title,
            "cluster_definition": cluster_definition,
            "reviews": reviews
        }

        open_questions = safe_dictionary_extraction(
            required_fields, chain_input, questions_chain,
            2, 2.0
        )

        open_questions['Cluster ID'] = cluster
        cluster_questions.append(open_questions)

        cluster_questions_df = pd.DataFrame(cluster_questions)
        cluster_questions_df.to_csv(checkpoint_file, index=False)

    # Add open questions column
    cluster_definitions_df['Open Questions'] = cluster_questions_df['Open Questions']
    cluster_definitions_df.to_csv(output_file, index=False)
    print(f"\nFinal file saved in: {output_file}")


**Trends Extraction (In right position)**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.parsing import parse_directories
from src.utils.semantic import load_configurations, safe_dictionary_extraction

from dotenv import load_dotenv

# Add the 'src' directory to the Python path
# This allows importing custom project modules (e.g., src.utils, src.classes) without import errors
sys.path.append('/content/drive/MyDrive/NeuroScape/src')

# Load environment keys
load_dotenv("/content/drive/MyDrive/NeuroScape/keys.env")
BASEPATH = os.environ['BASEPATH']

# Prompt template
TRENDS_PROMPT = PromptTemplate(
    input_variables=["title", "year", "old_abstracts", "recent_abstracts"],
    template="""
You are an expert in neuroscience and scientific text analysis.

You are provided with older abstracts (pre-{year}) and recent abstracts (post-{year}) of the cluster "{title}".

Your task is to compare the abstracts and identify the following:
- Emerging thematic trends: New or increasingly emphasized topics (as well as theories) that appear predominantly in the recent set.
- Emerging methodological trends: New tools, techniques, or approaches introduced or significantly gaining traction in the recent set.
- Declining themes: Topics or themes that were prominent in the older set but are less emphasized or absent in the recent set.
- Declining methodological approaches: Tools, techniques, or approaches that were commonly used in the older set but are now less represented or obsolete.
- Provide your analysis in a structured JSON format.

### STRICT INSTRUCTIONS:
1. Output must be ONLY valid JSON.
2. Do not include any introduction, explanation, commentary, or markdown.
3. The output must begin with "{{" and end with "}}".
4. Use the exact keys:
    - "Emerging Themes": "Semicolon separated list of emerging thematic trends with brief descriptions",
    - "Emerging Methodological Approaches": "Semicolon separated list of emerging methods or techniques with brief descriptions",
    - "Declining Themes": "Semicolon separated list of themes with brief descriptions that are less emphasized or absent",
    - "Declining Methodological Approaches": "Semicolon separated list of older methods or techniques with brief descriptions that are now less used"
5. Each value must be a single string with items separated by semicolons.
6. If no information is available for a field, return an empty string "".

### Older abstracts:
{old_abstracts}

### Recent abstracts:
{recent_abstracts}
"""
)

if __name__ == '__main__':
    configurations = load_configurations()
    directories = parse_directories()

    num_abstracts_per_set = 2 // 2
    required_fields = configurations['trends']['required_fields']
    recent_cutoff = configurations['trends']['recent_cutoff']
    older_cutoff = configurations['trends']['older_cutoff']

    # LLM Ollama
    llm = ChatOllama(
        model="orca-mini:13b",
        temperature=0.0
    )
    trends_chain = TRENDS_PROMPT | llm | SimpleJsonOutputParser()

    # Directories and files
    csv_directory = os.path.join('/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience')
    article_csv_file = 'articles_with_citation_rate.csv'
    article_df = pd.read_csv(os.path.join(csv_directory, article_csv_file))

    cluster_csv_file = 'clusters_defined_distinguished_questions.csv'
    cluster_df = pd.read_csv(os.path.join(csv_directory, cluster_csv_file))
    output_file = os.path.join(csv_directory, cluster_csv_file.replace('.csv', '_trends.csv'))

    checkpoint_path = os.path.join(BASEPATH, directories['internal']['checkpoints'])
    checkpoint_file = os.path.join(checkpoint_path, 'trends_checkpoint.csv')

    if os.path.exists(checkpoint_file):
        cluster_trends_df = pd.read_csv(checkpoint_file)
        cluster_trends = cluster_trends_df.to_dict('records')
        processed_clusters = set(cluster_trends_df['Cluster ID'].values.tolist())
        print(f'{len(processed_clusters)} processed clusters.')
    else:
        cluster_trends = []
        processed_clusters = set()
        print("No checkpoint found, starting from scratch.")

    # Selecionar apenas artigos de pesquisa
    research_articles = article_df[article_df['Type'] == 'Research']

    unique_labels = sorted(research_articles['Cluster ID'].unique())

    for label in tqdm(unique_labels):
        if label in processed_clusters:
            continue

        # Old abstracts
        older_abstracts_arr = research_articles[
            (research_articles['Cluster ID'] == label) &
            (research_articles['Year'] >= older_cutoff) &
            (research_articles['Year'] < recent_cutoff)
        ]['Abstract'].values

        num_old = min(num_abstracts_per_set, len(older_abstracts_arr))
        older_abstracts = '\n\n'.join(np.random.choice(older_abstracts_arr, num_old, replace=False))

        # Newer abstracts
        recent_df = research_articles[
            (research_articles['Cluster ID'] == label) &
            (research_articles['Year'] >= recent_cutoff)
        ].sort_values('Citation Rate', ascending=False)

        recent_abstracts_arr = recent_df['Abstract'].values[:num_abstracts_per_set]
        recent_abstracts = '\n\n'.join(recent_abstracts_arr)

        cluster_title = cluster_df.loc[label, 'Title']

        chain_input = {
            "title": cluster_title,
            "year": recent_cutoff,
            "old_abstracts": older_abstracts,
            "recent_abstracts": recent_abstracts
        }

        cluster_trends_dict = safe_dictionary_extraction(
            required_fields, chain_input, trends_chain,
            2, 2.0
        )

        cluster_trends_dict['Cluster ID'] = label
        cluster_trends.append(cluster_trends_dict)
        processed_clusters.add(label)

        # Save checkpoint
        pd.DataFrame(cluster_trends).to_csv(checkpoint_file, index=False)

    # Merge with cluster definitions
    cluster_trends_df = pd.DataFrame(cluster_trends)
    cluster_trends_df = cluster_df.merge(cluster_trends_df, on='Cluster ID')

    # Save final file
    cluster_trends_df.to_csv(output_file, index=False)
    print(f"\nArquivo final salvo em: {output_file}")


**Dimensions Extraction**

In [ ]:
import os
import time
import tomllib
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

from langchain_community.chat_models import ChatOllama
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers.json import SimpleJsonOutputParser

from src.utils.parsing import parse_directories
from src.utils.hypersphere import get_centroids
from src.utils.semantic import load_configurations, safe_dictionary_extraction, get_abstract_strings
from src.utils.load_and_save import load_embedding_shards, align_to_df

from dotenv import load_dotenv

import sys

# Add the 'src' directory to the Python path
# This allows importing custom project modules (e.g., src.utils, src.classes) without import errors
sys.path.append('/content/drive/MyDrive/NeuroScape/src')

# Load environment keys
load_dotenv("/content/drive/MyDrive/NeuroScape/keys.env")
BASEPATH = os.environ['BASEPATH']

# Define prompt template
DIMENSIONS_PROMPT = PromptTemplate(
    input_variables=["title", "abstracts"],
    template="""
            You are an expert in neuroscience and scientific text analysis.
            You are provided with a list of neuroscientific abstracts that belong to the cluster "{title}".

            Your task is to identify **key neuroscience dimensions** that best describe the cluster, guided by these 9 dimensions:

            1. Appliedness: Basic science vs applied (translational, clinical, technology development).
            2. Modality: Sensory and/or motor modality (visual, auditory, somatosensory, motor, multimodal, etc.).
            3. Spatiotemporal Scale: Spatial scale (molecular, cellular, circuit, systems, whole-brain) and temporal scale (milliseconds to lifetime).
            4. Cognitive Complexity: From low-level (sensory, motor) to high-level (language, decision making, social cognition).
            5. Species: Human, non-human primate, rodent, drosophila, zebrafish, C. elegans, etc.
            6. Theory Engagement: Theory-driven vs data-driven.
            7. Theory Scope: Specific mechanisms vs broad frameworks (Predictive Coding, Free Energy Principle, Global Workspace, IIT, etc.). Can be multiple or none.
            8. Methodological Approach: Experimental, computational, modeling, data analysis, meta-analysis, etc. Identify specific methods if present (fMRI, EEG, optogenetics, etc.).
            9. Interdisciplinarity: Degree of integration with other fields (medicine, psychology, CS, physics, engineering, philosophy, etc.).

            ---
            ### STRICT OUTPUT INSTRUCTIONS
            1. Output MUST be **only valid JSON**.
            2. Do NOT include any introduction, explanation, commentary, or extra text.
            3. Output MUST start with `{{` and end with `}}`.
            4. Use exactly these 9 keys:

                - "Dimension 1 - Appliedness"
                - "Dimension 2 - Modality"
                - "Dimension 3 - Spatiotemporal Scale"
                - "Dimension 4 - Cognitive Complexity"
                - "Dimension 5 - Species"
                - "Dimension 6 - Theory Engagement"
                - "Dimension 7 - Theory Scope"
                - "Dimension 8 - Methodological Approach"
                - "Dimension 9 - Interdisciplinarity"

            5. Each value MUST be a short string summarizing the abstracts.

            ---
            ### STRICT OUTPUT INSTRUCTIONS
            IMPORTANT: Return ONLY JSON. Do NOT include any text. Use exactly these 9 keys:
              "Dimension 1 - Appliedness": "Mostly basic neuroscience with some translational focus",
              "Dimension 2 - Modality": "Visual and auditory systems",
              "Dimension 3 - Spatiotemporal Scale": "Circuit and systems level across milliseconds to seconds",
              "Dimension 4 - Cognitive Complexity": "Low to intermediate cognitive processes",
              "Dimension 5 - Species": "Rodents and humans",
              "Dimension 6 - Theory Engagement": "Primarily hypothesis-driven",
              "Dimension 7 - Theory Scope": "No unifying framework explicitly stated",
              "Dimension 8 - Methodological Approach": "Electrophysiology and fMRI",
              "Dimension 9 - Interdisciplinarity": "Moderate; integrates neuroscience and psychology"
            Any deviation from this format will break parsing.

            ---
            ### ABSTRACTS
            {abstracts}""",
)

if __name__ == '__main__':
    configurations = load_configurations()
    required_fields = configurations['dimensions']['required_fields']
    directories = parse_directories()

    # Substituting OpenAI -> Ollama
    llm = ChatOllama(
        model="orca-mini:13b",
        temperature=0.0
    )

    dimensions_chain = DIMENSIONS_PROMPT | llm | SimpleJsonOutputParser()

     # Load the CSV file
    print("Loading datasets...")
    csv_directory = os.path.join('/content/drive/MyDrive/NeuroScape/output/tratados/neuroscience')
    article_csv_file = 'articles_merged_cleaned_filtered_clustered.csv'
    article_df = pd.read_csv(os.path.join(csv_directory, article_csv_file))
    article_df = article_df[article_df['Type'] == 'Research']
    print(f"Filtered articles (Research): {len(article_df)}")

    # Define the output path
    cluster_csv_file = 'clusters_defined_distinguished_questions_trends.csv'
    cluster_df = pd.read_csv(os.path.join(csv_directory, cluster_csv_file))
    print(f"Analyzed clusters: {len(cluster_df)}")
    output_file = os.path.join(
        csv_directory, cluster_csv_file.replace('.csv', '_assessed.csv'))
    checkpoints_file = os.path.join('/content/drive/MyDrive/NeuroScape/output/checkpoints/clusters/dimensions extraction',
                                    'assessment_checkpoint.csv')

    if os.path.exists(checkpoints_file):
        cluster_dimensions_df = pd.read_csv(checkpoints_file)
        cluster_dimensions = cluster_dimensions_df.to_dict('records')
        processed_clusters = set(
            cluster_dimensions_df['Cluster ID'].values.tolist())

        print(f'{len(processed_clusters)} clusters have been processed.')
    else:
        cluster_dimensions = []
        processed_clusters = set()

    # Load the embeddings and PMIDs
    print("Loading embeddings...")
    shard_directory = os.path.join(
        BASEPATH, directories['internal']['intermediate']['hdf5']['neuro'])
    files = glob(os.path.join(shard_directory, '*.h5'))
    print(f"Found shards: {len(files)}")
    embeddings, pmids = load_embedding_shards(files)
    print(f"Embeddings shape: {embeddings.shape}, PMIDs: {len(pmids)}")

    # Align the embeddings and PMIDs to the DataFrame
    aligned_embeddings, aligned_pmids = align_to_df(embeddings, pmids,
                                                    article_df)
    print(f"Embeddings aligned: {aligned_embeddings.shape}, PMIDs aligned: {len(aligned_pmids)}")


    # Free memory
    del embeddings
    del pmids

    # Get the centroids of the clusters
    centroids = get_centroids(aligned_embeddings,
                              article_df['Cluster ID'].values)
    print(f"Calculated centroids: {centroids.shape}")
    centroid_similarities = centroids.dot(centroids.T) - np.eye(
        centroids.shape[0])

    # Get cluster labels
    labels = article_df['Cluster ID'].values

    # Get the abstracts
    abstracts = article_df['Abstract'].values

    for label, centroid in tqdm(enumerate(centroids)):

        if label in processed_clusters:
            print(f"Cluster {label} already processed, skipping...")
            continue

        print(f"\n Processing cluster {label}...")
        cluster_embeddings = aligned_embeddings[labels == label]
        cluster_abstracts = abstracts[labels == label]

        number_of_abstracts = min(
            2,
            len(cluster_abstracts))
        print(f"Number of selected abstracts: {number_of_abstracts}")

        similarity = centroid.dot(cluster_embeddings.T)
        top_indices = np.argsort(similarity)[::-1][:number_of_abstracts]
        print(f"Top indexes: {top_indices[:10]}")

        cluster_abstracts = '\n\n'.join(cluster_abstracts[top_indices])

        cluster_title = cluster_df.loc[label, 'Title']
        print(f"Cluster title: {cluster_title}")

        chain_input = {"title": cluster_title, "abstracts": cluster_abstracts}

        print("Calling LLM...")
        extracted_dict = safe_dictionary_extraction(
            required_fields, chain_input, dimensions_chain,
            2, 2.0)
        print(f"Response: {extracted_dict}")

        # Join the keys as a single string with each key as a title and a line break between them
        cluster_dimensions_dict = {
            'Dimensions':
            '\n'.join(
                [f'{key}: {value}' for key, value in extracted_dict.items()])
        }

        cluster_dimensions_dict['Cluster ID'] = label
        cluster_dimensions.append(cluster_dimensions_dict)
        processed_clusters.add(label)

        cluster_dimensions_df = pd.DataFrame(cluster_dimensions)
        cluster_dimensions_df.to_csv(checkpoints_file, index=False)

        print(cluster_dimensions)
        break

    # Convert the cluster definitions to a DataFrame
    cluster_dimensions_df = pd.DataFrame(cluster_dimensions)

    # Merge the cluster definitions with the existing cluster definitions
    cluster_dimensions_df = cluster_df.merge(cluster_dimensions_df,
                                             on='Cluster ID')

    # Save the cluster definitions to a CSV file
    cluster_dimensions_df.to_csv(output_file, index=False)